# gcsfs smoke test

Smoke test for `gcsfs`, the fsspec filesystem for Google Cloud Storage. Every example runs offline:
the filesystem is constructed with anonymous credentials and no request is sent to Cloud Storage.

## Example 1: Protocol registration and URL parsing

In [ ]:
# Example 1 — fsspec protocol registration, URL parsing and anonymous credentials, all offline.
#
# pandas and dask resolve gs:// URLs through fsspec.core.url_to_fs, which looks the protocol up in
# fsspec's registry and instantiates gcsfs.
#
# `issubclass`, not `is`: recent gcsfs releases register `ExtendedGcsFileSystem`, a subclass of
# GCSFileSystem, for `gs`/`gcs`. get_filesystem_class raises if no class is registered at all.
import fsspec
import gcsfs

for protocol in ("gs", "gcs"):
    resolved = fsspec.get_filesystem_class(protocol)
    assert issubclass(resolved, gcsfs.GCSFileSystem), (protocol, resolved)

fs, path = fsspec.core.url_to_fs("gs://bucket/a/b.csv", token="anon", skip_instance_cache=True)
assert isinstance(fs, gcsfs.GCSFileSystem), type(fs)
assert path == "bucket/a/b.csv", path
assert fs.split_path("gs://bucket/a/b.csv") == ("bucket", "a/b.csv", None)
assert fs.split_path("bucket/a/b.csv") == ("bucket", "a/b.csv", None)
assert fs.split_path("gs://bucket/") == ("bucket", "", None)  # a bare bucket is a bucket, never a key
# token="anon" resolves to anonymous credentials through google-auth.
assert fs.credentials.method == "anon", fs.credentials.method
print("gs/gcs ->", type(fs).__name__, "| url_to_fs ->", path, "| credentials:", fs.credentials.method)

## Example 2: Path helpers

In [ ]:
# Example 2 — gcsfs path helpers: protocol stripping, parents, object generations and download URLs.
# Pure string handling on an anonymous filesystem; no GCS request is made.
import gcsfs

GCS = gcsfs.GCSFileSystem
assert GCS._strip_protocol("gs://bucket/a/b.csv") == "bucket/a/b.csv"
assert GCS._strip_protocol("gcs://bucket/a/b.csv") == "bucket/a/b.csv"
assert GCS._parent("gs://bucket/a/b.csv") == "bucket/a"
assert GCS._parent("bucket/a") == "bucket"

fs = GCS(token="anon", skip_instance_cache=True)
assert fs.unstrip_protocol("bucket/a/b.csv") == "gs://bucket/a/b.csv"

# With version_aware=True a "#<generation>" suffix selects one object generation;
# without it, the suffix stays part of the object name.
versioned = GCS(token="anon", version_aware=True, skip_instance_cache=True)
assert versioned.split_path("gs://bucket/a/b.csv#123") == ("bucket", "a/b.csv", "123")
assert fs.split_path("gs://bucket/a/b.csv#123") == ("bucket", "a/b.csv#123", None)

# url() builds the JSON-API media download URL, with the object name percent-encoded.
download = fs.url("gs://bucket/dir/a b.csv")
assert download.endswith("/download/storage/v1/b/bucket/o/dir%2Fa%20b.csv?alt=media"), download
assert fs.url("gs://bucket/a.csv", generation=7).endswith("?alt=media&generation=7")
print("path helpers OK:", download.split("/download/")[1])

## Example 3: HTTP status to exception mapping

In [ ]:
# Example 3 — the HTTP-status -> Python-exception mapping, and the retry policy built on it.
#
# gcsfs.retry.validate_response turns a GCS response status into a Python exception: a missing object
# raises FileNotFoundError, which callers such as pandas and dask catch. It is a pure function of
# (status, body, path), so it runs with no network and no credentials.
#
# The exception type is compared exactly: FileNotFoundError and FileExistsError are both OSError
# subclasses, so an isinstance(exc, OSError) check would not tell the cases apart.
from gcsfs.retry import HttpError, is_retriable, validate_response

validate_response(200, b"ok", "gs://b/k")  # a success must not raise at all

for status, body, expected in [
    (404, b"", FileNotFoundError),
    (403, b'{"error":{"message":"denied"}}', OSError),
    (412, b"", FileExistsError),
    (500, b'{"error":{"message":"boom","code":500}}', HttpError),
]:
    raised = None
    try:
        validate_response(status, body, "gs://b/k")
    except BaseException as error:  # noqa: BLE001 - the type IS the assertion
        raised = error
    assert raised is not None, f"HTTP {status} did not raise"
    assert type(raised) is expected, (status, type(raised).__name__)

assert is_retriable(HttpError({"code": 429, "message": "slow down"}))
assert is_retriable(HttpError({"code": 500, "message": "boom"}))
assert not is_retriable(FileNotFoundError("gs://b/missing"))  # a 404 must never be retried
print("status->exception contract OK for 200/404/403/412/500 + retry policy")

## Example 4: Checksum validation

In [ ]:
# Example 4 — consistency checking: gcsfs hashes the bytes it transfers and compares the digest with
# the object metadata GCS returns. The metadata here is built locally from known bytes.
import base64
import hashlib

from gcsfs.checkers import ChecksumError, ConsistencyChecker, MD5Checker, get_consistency_checker

payload = b"hello gcsfs"
md5_b64 = base64.b64encode(hashlib.md5(payload).digest()).decode()

checker = get_consistency_checker("md5")
assert isinstance(checker, MD5Checker), type(checker)
checker.update(payload[:5])
checker.update(payload[5:])  # fed in chunks, as a streamed download would be
checker.validate_json_response({"md5Hash": md5_b64})  # matching digest: no exception
checker.validate_headers({"X-Goog-Hash": f"crc32c=AAAAAA==,md5={md5_b64}"})

try:
    checker.validate_json_response({"md5Hash": base64.b64encode(hashlib.md5(b"other").digest()).decode()})
except ChecksumError:
    pass
else:
    raise AssertionError("a mismatched MD5 digest did not raise ChecksumError")

# consistency=None selects the base checker, which accepts anything.
assert type(get_consistency_checker(None)) is ConsistencyChecker
print("md5 consistency check OK:", md5_b64)